# Calibration Importance Analysis
## Does Individual Calibration Matter?

Upload two CSV files from the Calibration Importance Test:
- **File 1**: Subject using their **own** calibration
- **File 2**: Subject using **someone else's** calibration

Throughput is calculated using the **ISO 9241-9** standard (directional projection effective width).

In [ ]:
# Install dependencies (if needed)
!pip install -q pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab import files

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

In [ ]:
# ============================
# UPLOAD YOUR TWO CSV FILES
# ============================
print("Upload the FIRST file (subject using their OWN calibration):")
uploaded1 = files.upload()
file1_name = list(uploaded1.keys())[0]

print("\nUpload the SECOND file (subject using SOMEONE ELSE's calibration):")
uploaded2 = files.upload()
file2_name = list(uploaded2.keys())[0]

df_own = pd.read_csv(file1_name)
df_other = pd.read_csv(file2_name)

# Extract labels
label_own = df_own['calibrationSource'].iloc[0]
label_other = df_other['calibrationSource'].iloc[0]

print(f"\n--- File 1 ---")
print(f"  Calibration: {label_own}")
print(f"  Trials: {len(df_own)}")
print(f"\n--- File 2 ---")
print(f"  Calibration: {label_other}")
print(f"  Trials: {len(df_other)}")

In [ ]:
# ================================================
# ISO 9241-9 Throughput Calculation (Directional Projection)
# ================================================

def compute_iso_metrics(df):
    """Compute per-layout ISO 9241-9 aggregate metrics using directional projection."""
    results = []
    
    for layout_idx in sorted(df['layoutIndex'].unique()):
        layout_df = df[(df['layoutIndex'] == layout_idx) & (df['movementTime'].notna()) & (df['movementTime'] > 0)]
        
        if len(layout_df) == 0:
            continue
        
        # Mean movement time
        mean_mt = layout_df['movementTime'].mean()
        
        # Effective amplitude (mean of actual amplitudes)
        Ae = layout_df['actualAmplitude'].mean()
        
        # ISO 9241-9 directional projection with corrected target positions
        theta_rad = np.radians(layout_df['direction'].astype(float))
        first_trial = layout_df[layout_df['trialInLayout'] == 0].iloc[0]
        center_x, center_y = first_trial['startX'], first_trial['startY']
        correct_target_x = center_x + layout_df['amplitude'] * np.cos(theta_rad)
        correct_target_y = center_y + layout_df['amplitude'] * np.sin(theta_rad)
        dx = layout_df['selectionX'] - correct_target_x
        dy = layout_df['selectionY'] - correct_target_y
        projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
        SDx = projections.std(ddof=1)
        
        # Effective width
        We = 4.133 * SDx
        
        # Effective index of difficulty
        IDe = np.log2(Ae / We + 1)
        
        # Throughput
        TP = IDe / mean_mt
        
        # Target size & amplitude in percentages (normalize back)
        target_size_px = layout_df['targetSize'].iloc[0]
        amplitude_px = layout_df['amplitude'].iloc[0]
        
        results.append({
            'layoutIndex': layout_idx,
            'targetSize': target_size_px,
            'amplitude': amplitude_px,
            'meanMT': mean_mt,
            'Ae': Ae,
            'SDx': SDx,
            'We': We,
            'IDe': IDe,
            'TP': TP,
            'n': len(layout_df)
        })
    
    return pd.DataFrame(results)

metrics_own = compute_iso_metrics(df_own)
metrics_other = compute_iso_metrics(df_other)

print("=" * 70)
print(f"OWN CALIBRATION: {label_own}")
print("=" * 70)
print(metrics_own[['layoutIndex', 'targetSize', 'amplitude', 'meanMT', 'Ae', 'We', 'IDe', 'TP']].to_string(index=False, float_format='%.3f'))
print(f"\n  Average MT:  {metrics_own['meanMT'].mean():.3f}s")
print(f"  Average TP:  {metrics_own['TP'].mean():.3f} bps")

print(f"\n{'=' * 70}")
print(f"OTHER CALIBRATION: {label_other}")
print("=" * 70)
print(metrics_other[['layoutIndex', 'targetSize', 'amplitude', 'meanMT', 'Ae', 'We', 'IDe', 'TP']].to_string(index=False, float_format='%.3f'))
print(f"\n  Average MT:  {metrics_other['meanMT'].mean():.3f}s")
print(f"  Average TP:  {metrics_other['TP'].mean():.3f} bps")

In [ ]:
# ================================================
# SUMMARY COMPARISON
# ================================================

avg_mt_own = metrics_own['meanMT'].mean()
avg_mt_other = metrics_other['meanMT'].mean()
avg_tp_own = metrics_own['TP'].mean()
avg_tp_other = metrics_other['TP'].mean()

mt_diff_pct = (avg_mt_other - avg_mt_own) / avg_mt_own * 100
tp_diff_pct = (avg_tp_other - avg_tp_own) / avg_tp_own * 100

print("\n" + "=" * 60)
print("CALIBRATION IMPORTANCE - SUMMARY")
print("=" * 60)
print(f"{'Metric':<25} {'Own Calib':>12} {'Other Calib':>12} {'Diff':>10}")
print("-" * 60)
print(f"{'Avg Movement Time (s)':<25} {avg_mt_own:>12.3f} {avg_mt_other:>12.3f} {mt_diff_pct:>+9.1f}%")
print(f"{'Avg Throughput (bps)':<25} {avg_tp_own:>12.3f} {avg_tp_other:>12.3f} {tp_diff_pct:>+9.1f}%")
print("=" * 60)

if avg_tp_own > avg_tp_other:
    print(f"\n>> Own calibration is BETTER: {abs(tp_diff_pct):.1f}% higher throughput, {abs(mt_diff_pct):.1f}% faster")
else:
    print(f"\n>> Other calibration is BETTER: {abs(tp_diff_pct):.1f}% higher throughput, {abs(mt_diff_pct):.1f}% faster")

In [ ]:
# ================================================
# VISUALIZATION 1: Bar Charts (MT & TP)
# ================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2196F3', '#FF5722']
labels = ['Own Calibration', "Other's Calibration"]

# --- Movement Time ---
ax = axes[0]
bars = ax.bar(labels, [avg_mt_own, avg_mt_other], color=colors, width=0.5, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Movement Time (s)')
ax.set_title('Average Movement Time')
for bar, val in zip(bars, [avg_mt_own, avg_mt_other]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}s', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylim(0, max(avg_mt_own, avg_mt_other) * 1.25)

# --- Throughput ---
ax = axes[1]
bars = ax.bar(labels, [avg_tp_own, avg_tp_other], color=colors, width=0.5, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Throughput (bps)')
ax.set_title('Average Throughput (ISO 9241-9)')
for bar, val in zip(bars, [avg_tp_own, avg_tp_other]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylim(0, max(avg_tp_own, avg_tp_other) * 1.25)

plt.suptitle('Calibration Importance: Own vs Other', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ================================================
# VISUALIZATION 2: Per-Layout Comparison
# ================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_layouts = len(metrics_own)
x = np.arange(n_layouts)
width = 0.35

# Layout labels (use target size % and amplitude %)
layout_labels = [f'S={row.targetSize:.0f}\nA={row.amplitude:.0f}' for _, row in metrics_own.iterrows()]

# --- Movement Time per layout ---
ax = axes[0]
bars1 = ax.bar(x - width/2, metrics_own['meanMT'], width, label='Own Calibration', color='#2196F3', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, metrics_other['meanMT'], width, label="Other's Calibration", color='#FF5722', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Layout (Size / Amplitude in px)')
ax.set_ylabel('Mean Movement Time (s)')
ax.set_title('Movement Time by Layout')
ax.set_xticks(x)
ax.set_xticklabels(layout_labels, fontsize=8)
ax.legend()

# --- Throughput per layout ---
ax = axes[1]
bars1 = ax.bar(x - width/2, metrics_own['TP'], width, label='Own Calibration', color='#2196F3', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, metrics_other['TP'], width, label="Other's Calibration", color='#FF5722', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Layout (Size / Amplitude in px)')
ax.set_ylabel('Throughput (bps)')
ax.set_title('Throughput by Layout (ISO 9241-9)')
ax.set_xticks(x)
ax.set_xticklabels(layout_labels, fontsize=8)
ax.legend()

plt.suptitle('Per-Layout Comparison: Own vs Other Calibration', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ================================================
# VISUALIZATION 3: Trial-by-Trial Movement Times
# ================================================

fig, ax = plt.subplots(figsize=(14, 5))

valid_own = df_own[df_own['movementTime'].notna() & (df_own['movementTime'] > 0)]
valid_other = df_other[df_other['movementTime'].notna() & (df_other['movementTime'] > 0)]

ax.plot(valid_own['trialNumber'], valid_own['movementTime'], 'o-', color='#2196F3',
        markersize=4, linewidth=1, alpha=0.7, label='Own Calibration')
ax.plot(valid_other['trialNumber'], valid_other['movementTime'], 's-', color='#FF5722',
        markersize=4, linewidth=1, alpha=0.7, label="Other's Calibration")

ax.axhline(y=avg_mt_own, color='#2196F3', linestyle='--', alpha=0.5, label=f'Own avg: {avg_mt_own:.3f}s')
ax.axhline(y=avg_mt_other, color='#FF5722', linestyle='--', alpha=0.5, label=f'Other avg: {avg_mt_other:.3f}s')

# Add layout boundaries
for i in range(1, 6):
    ax.axvline(x=i*8 + 0.5, color='gray', linestyle=':', alpha=0.3)

ax.set_xlabel('Trial Number')
ax.set_ylabel('Movement Time (s)')
ax.set_title('Trial-by-Trial Movement Times')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# ================================================
# VISUALIZATION 4: Selection Error Distribution
# ================================================

fig, ax = plt.subplots(figsize=(10, 5))

def compute_corrected_error(df):
    theta_rad = np.radians(df['direction'].astype(float))
    errors = []
    for layout_idx in df['layoutIndex'].unique():
        layout_df = df[df['layoutIndex'] == layout_idx]
        first_trial = layout_df[layout_df['trialInLayout'] == 0].iloc[0]
        cx, cy = first_trial['startX'], first_trial['startY']
        correct_tx = cx + layout_df['amplitude'] * np.cos(np.radians(layout_df['direction'].astype(float)))
        correct_ty = cy + layout_df['amplitude'] * np.sin(np.radians(layout_df['direction'].astype(float)))
        err = np.sqrt((layout_df['selectionX'] - correct_tx)**2 + (layout_df['selectionY'] - correct_ty)**2)
        errors.append(err)
    return pd.concat(errors)

err_own = compute_corrected_error(df_own)
err_other = compute_corrected_error(df_other)

ax.hist(err_own, bins=20, alpha=0.6, color='#2196F3',
        label=f'Own Calib (mean: {err_own.mean():.1f}px)', edgecolor='black', linewidth=0.5)
ax.hist(err_other, bins=20, alpha=0.6, color='#FF5722',
        label=f'Other Calib (mean: {err_other.mean():.1f}px)', edgecolor='black', linewidth=0.5)

ax.set_xlabel('Selection Error (px)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Selection Errors')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ================================================
# VISUALIZATION 5: Fitts' Law Regression
# ================================================

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(metrics_own['IDe'], metrics_own['meanMT'], color='#2196F3', s=80,
           zorder=5, edgecolors='black', linewidth=0.5, label='Own Calibration')
ax.scatter(metrics_other['IDe'], metrics_other['meanMT'], color='#FF5722', s=80,
           zorder=5, edgecolors='black', linewidth=0.5, marker='s', label="Other's Calibration")

# Regression lines
for metrics, color, name in [(metrics_own, '#2196F3', 'Own'), (metrics_other, '#FF5722', 'Other')]:
    z = np.polyfit(metrics['IDe'], metrics['meanMT'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(metrics['IDe'].min() - 0.2, metrics['IDe'].max() + 0.2, 100)
    ax.plot(x_line, p(x_line), '--', color=color, alpha=0.5)
    # R-squared
    ss_res = np.sum((metrics['meanMT'] - p(metrics['IDe']))**2)
    ss_tot = np.sum((metrics['meanMT'] - metrics['meanMT'].mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    ax.text(x_line[-1], p(x_line[-1]), f' R²={r2:.2f}', color=color, fontsize=9)

ax.set_xlabel('Effective Index of Difficulty (IDe, bits)')
ax.set_ylabel('Mean Movement Time (s)')
ax.set_title("Fitts' Law: MT vs IDe")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ================================================
# FULL DETAILED TABLE
# ================================================

print("\n" + "=" * 80)
print("DETAILED PER-LAYOUT METRICS")
print("=" * 80)

combined = metrics_own[['layoutIndex', 'targetSize', 'amplitude', 'meanMT', 'TP', 'IDe', 'We', 'n']].copy()
combined.columns = ['Layout', 'TgtSize', 'Amp', 'MT_own', 'TP_own', 'IDe_own', 'We_own', 'n']
combined['MT_other'] = metrics_other['meanMT'].values
combined['TP_other'] = metrics_other['TP'].values
combined['IDe_other'] = metrics_other['IDe'].values
combined['We_other'] = metrics_other['We'].values
combined['MT_diff%'] = (combined['MT_other'] - combined['MT_own']) / combined['MT_own'] * 100
combined['TP_diff%'] = (combined['TP_other'] - combined['TP_own']) / combined['TP_own'] * 100

print(combined.to_string(index=False, float_format='%.3f'))
print(f"\nOwn calibration wins in {(combined['TP_own'] > combined['TP_other']).sum()}/{len(combined)} layouts for throughput")
print(f"Own calibration wins in {(combined['MT_own'] < combined['MT_other']).sum()}/{len(combined)} layouts for movement time")